In [ ]:
import warnings
warnings.filterwarnings('ignore')#ometimes things aren't broken enough to crash but something is worth flagging like using an outdated function

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv(r'C:\Users\sahil\Desktop\Bike Sharing Prediction\day.csv')

In [ ]:
df.rename(columns={'instant': 'SrNo', 'dteday': 'date','cnt':'count'}, inplace=True)

In [ ]:
df

In [ ]:
df.shape #how many rows and columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#Assigning string values to season as it will misindicate the machine
#Only one weight w_season for all 4 seasons. The model cannot learn that spring and winter are both low but summer and fall are high. 
#astype treats it as category

In [ ]:
df.loc[(df['season'] == 1), 'season'] = 'spring'
df.loc[(df['season'] == 2), 'season'] = 'summer'
df.loc[(df['season'] == 3), 'season'] = 'fall'
df.loc[(df['season'] == 4), 'season'] = 'winter'

In [ ]:
df['yr'].astype('category').value_counts()

In [ ]:
month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}

df['mnth'] = df['mnth'].map(month_map).astype('category')

In [ ]:
df['mnth'].astype('category').value_counts()

In [ ]:
weekday_map = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thur', 5:'Fri', 6:'Sat',0:'Sun'}
df['weekday'] = df['weekday'].map(weekday_map).astype('category')

In [ ]:
df['weekday'].astype('category').value_counts()

In [ ]:
weather_map = {1:'Clear', 2:'Mist', 3:'Light_Rain', 4:'Heavy_Rain'}
df['weathersit'] = df['weathersit'].map(weather_map).astype('category')

In [ ]:
df['weathersit'].astype('category').value_counts()

In [ ]:
# Temperature
sns.distplot(df['temp'])
plt.show()

In [ ]:
# Actual Temperature
sns.distplot(df['atemp'])
plt.show()

In [ ]:
#wind speed
sns.displot(df['windspeed'])
plt.show()

In [ ]:
#count
sns.displot(df['count'])
plt.show()

In [ ]:
#converting date to datetime format

In [ ]:
df['date'] = df['date'].astype('datetime64[ns]')

In [ ]:
dc=df.select_dtypes(exclude=['float64','datetime64[ns]','int64'])
dc.columns
dc

In [ ]:
plt.figure(figsize=(20,20))
plt.subplot(3,3,1)
sns.boxplot(x = 'season', y = 'count', data=df)
plt.subplot(3,3,2)
sns.boxplot(x = 'mnth', y = 'count', data=df)
plt.subplot(3,3,3)
sns.boxplot(x = 'weathersit', y = 'count', data=df)
plt.subplot(3,3,4)
sns.boxplot(x = 'holiday', y = 'count', data=df)
plt.subplot(3,3,5)
sns.boxplot(x = 'weekday', y = 'count', data=df)
plt.subplot(3,3,6)
sns.boxplot(x = 'workingday', y = 'count', data=df)
plt.subplot(3,3,7)
sns.boxplot(x = 'yr', y = 'count', data=df)
plt.show()

In [ ]:
intVarlist = ["casual", "registered", "count"]

for var in intVarlist:
    df[var] = df[var].astype("float")

dataset_numeric = df.select_dtypes(include=['float64'])
dataset_numeric.head()

sns.pairplot(dataset_numeric)
plt.show()
#Show me every possible relationship between my columns, all at once, visually corelation with each other
#if this feature goes up what happens to other feature that is shown in pairplot

In [ ]:
cor = dataset_numeric.corr()
cor

# heatmap
mask = np.array(cor)
mask[np.tril_indices_from(mask)] = False
fig, ax = plt.subplots()
fig.set_size_inches(10, 10)
sns.heatmap(cor, mask=mask, vmax=1, square=True, annot=True)
#summary of pairplot 

In [ ]:
df.drop('atemp', axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
dc.head()

In [ ]:
df_dummies=pd.get_dummies(dc,drop_first=True,dtype=int
)#gives independent w value for subfeature of feature (wx+b) drop_first to remove first column of dummies to avoid
df_dummies

In [ ]:
df.drop(list(dc.columns),axis=1,inplace=True)
df

In [ ]:
#concat with dummy dataset 
df=pd.concat([df,df_dummies],axis=1)

In [ ]:
df.drop(['SrNo','date'], axis=1, inplace=True,errors='ignore')#axis=0-drop ROWS axis=1-drop COLUMNS
#inplace=true makes changes direcly to original whereas false does it in copy


In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split

y = df['count']                   
X = df.drop('count', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train)#it returns numpy we gotta convert it to dataframe(panda)
X_train_sc = pd.DataFrame(X_train_sc, columns=X_train.columns)
X_test_sc = scaler.transform(X_test)  
X_test_sc = pd.DataFrame(X_test_sc, columns=X_test.columns)

In [ ]:
plt.figure(figsize=(30,30))
sns.heatmap(X_train_sc.corr(), annot=True, cmap="YlGnBu")

In [ ]:
X_train_sc = X_train_sc.drop(['casual','registered'], axis=1)
X_test_sc = X_test_sc.drop(['casual', 'registered'], axis=1)

 #data leakage casual+registered=1 so its same
#axis-1

In [ ]:
#stats model

In [ ]:
import statsmodels.api as sm #stats model is python library used for statistic analysis its 
#sklearn-predicts whereas statsmodel researches,analyzes
X_train_sm=sm.add_constant(X_train_sc)
X_test_sm = sm.add_constant(X_test_sc)
#adds a constant value of 1"s which is there in sklearn b*1
#or else for every point has to start from origin (b decides where it will slope starts from)

In [ ]:
y_train = y_train.reset_index(drop=True)#y indexing is not aligned
lr=sm.OLS(y_train,X_train_sm).fit()#this is just for us to see the summary of stats model not used to build model

In [ ]:
lr.params
lr.summary()


In [ ]:
#with lrsummary we see p values of each feature now instead of removing manually we use RFE(recursive feature eliminator)(eliminating least significantly)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

lm=LinearRegression()
rfe1 = RFE(estimator=lm, n_features_to_select=15)
#fiting with only 15 features
rfe1.fit(X_train_sc,y_train)


In [ ]:
print(rfe1.support_)

In [ ]:
col1 = X_train_sc.columns[rfe1.support_]  #eliminates false features

In [ ]:
X_train_rfe1 = X_train_sm[col1]           
X_test_rfe1 = X_test_sm[col1]            



In [ ]:
lm = LinearRegression()
lm.fit(X_train_rfe1, y_train)

In [ ]:
from sklearn.metrics import r2_score

y_pred = lm.predict(X_test_rfe1)
r2 = r2_score(y_test, y_pred)
print('R2 Score:', r2)

In [ ]:
 #VIF removes duplicate features hen two or more features are highly correlated with each other 
 # This is a problem because the model gets confusedcan't tell which feature is actually causing the target to change.
from statsmodels.stats.outliers_influence import variance_inflation_factor
a = X_train_rfe1.copy()
vif = pd.DataFrame()
vif["Features"] = a.columns
vif['VIF'] = [variance_inflation_factor(a.values, i) for i in range(a.shape[1])]
#what exactly is happening is for each feature eg temp VIF predict temp using other features(i=1 first feature)
#how vif is calculated is by taking each feature acalculating R2 score of that regression and then 
#if R2 is high then VIF will be high which means that feature is highly correlated 
#if R2 is low then VIF will be low which means that feature is not highly correlated 

In [ ]:
vif['VIF'] = vif['VIF'].round(3)  #rounding up
vif.sort_values(by='VIF', ascending=False)

In [ ]:
lm=LinearRegression()
rfe2 = RFE(estimator=lm, n_features_to_select=10)
col2=X_train_rfe1.columns[rfe2.fit(X_train_rfe1, y_train).support_]
col2

In [ ]:
X_train_rfe2 = X_train_rfe1[col2]
X_test_rfe2 = X_test_rfe1[col2]
X_train_rfe2=sm.add_constant(X_train_rfe2)
X_test_rfe2=sm.add_constant(X_test_rfe2)
lm2 = sm.OLS(y_train, X_train_rfe2).fit()
lm2.summary()


In [ ]:
# X_test_rfe2.info()

In [ ]:
y_pred = lm2.predict(X_test_rfe2)
r2 = r2_score(y_test, y_pred)
print('R2 Score:', r2)